<a href="https://colab.research.google.com/github/2403a52208-sudo/NLP/blob/main/2403A52208_B09_Assg15_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import libraries

In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np

Load ELMO Model

In [3]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Text Corpus

In [6]:
sentences = ["She deposited her paycheck at the bank",
             "He works at a bank downtown"]


Generate Embeddings

In [7]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

(2, 7, 1024)


Inspect Word Embedding (“bank”)

In [8]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bank"
idx1 = tokenized[0].index("bank")
idx2 = tokenized[1].index("bank")

# Extract embeddings
bank_emb_1 = embeddings[0][idx1]
bank_emb_2 = embeddings[1][idx2]

print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.44384375  0.26281542 -0.03348051  0.01434102 -0.07464592 -0.55423373
 -0.18376788  0.14679211  0.41404274  0.09957089], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.4375206   0.32943156 -0.09300089  0.04544412  0.16819152 -0.0100608
 -0.00214589  0.29912     0.54008144  0.43784827], shape=(10,), dtype=float32)


Compare Context (Cosine Similarity)

In [9]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bank' meanings:", sim)

Similarity between 'bank' meanings: 0.77928


Task - II

text corpus=["The room filled with bright light from the window""This suitcase is very light, easy to carry"]


BERT Model Implementation

In [10]:
sentence = ["The room filled with bright light from the window",
 "This suitcase is very light, easy to carry"]

In [11]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [12]:
inputs = preprocess(sentences)
print(inputs.keys())

dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


Generate Embeddings


In [13]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


Get Tokens (Important Step)

In [14]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[  101  2016 14140  2014  3477  5403  3600  2012  1996  2924   102     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0]
 [  101  2002  2573  2012  1037  2924  5116   102     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0 

Extract bat Embeddings


In [15]:
bat_emb_1 = embeddings[0][2]  # "bat" in first sentence
bat_emb_2 = embeddings[1][7]  # "bat" in second sentence
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.07500547 -0.81638026  0.4937751  -0.37808093  0.4427734   0.05464032
  0.04950104  0.37811115 -0.13197146 -0.16009405], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[ 0.81262296  0.15664959 -0.35921514  0.65484595 -0.49618474 -0.58152366
  0.2493905  -0.30181813  0.49611726  0.00212299], shape=(10,), dtype=float32)


Cosine Similarity

In [16]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.071778916


Text Classification Using ELMO+Naive Bayes


Import Libraries

In [17]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


Text Corpus

In [23]:
sentences = [
    "urgent call now",
    "free gift waiting",
    "how was your day",
    "let's go for lunch",
    "win a free ticket",
    "are you coming today"
]
labels = [1, 1, 0, 0, 1, 0]   # 1 = spam, 0 = normal

Load ELMo Model

In [24]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Generate ELMo Embeddings

In [25]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Shape: (6, 4, 1024)


Convert to Sentence Embeddings

In [26]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 1024)


Train-Test Split

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model

In [28]:
model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

Model Testing

In [29]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [0 0]
Actual: [1 1]


Model Evaluation

In [30]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.0


Prediction on New Text

In [31]:
new_sentence = ["Congratulations! You won a free ticket"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")

Prediction: Not Spam


Text Classification using BERT+NB

In [32]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [33]:
bert_inputs = preprocess(sentences)

In [34]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)


In [36]:
nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)

BERT Accuracy: 0.0
